# 07.2 - Bag of Words & TF-IDF

**Phase:** 07 - NLP

**Status:** VERIFIED

---

## 1. What Are We Solving?

Machine learning models cannot operate on raw text — they need numbers. **Bag of Words (BoW)** counts word occurrences per document. **TF-IDF** reweights these counts so rare, informative words score higher and common uninformative words score lower.

## 2. Why Does This Matter?

BoW and TF-IDF are the simplest, fastest, and most interpretable text-to-number mappings. They remain strong baselines for spam filtering, topic labeling, and search. Understanding their limitations (no word order, no meaning) motivates embeddings and transformers.

## 3. Prerequisites

- Unit 07.1 (preprocessing)
- Phase 05 (ML basics)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a document-term matrix with CountVectorizer and TfidfVectorizer
- Compute TF-IDF by hand and verify against sklearn
- Explain why TF-IDF beats raw counts
- Avoid data leakage (fit on train, transform on test)

## 5. Mental Model

BoW is a histogram of words per document. TF-IDF is a BoW where every word's count is multiplied by how rare it is across the corpus.

```text
Document = "the cat sat on the mat"
BoW      = {the:2, cat:1, sat:1, on:1, mat:1}
TF-IDF   = the low, cat high, mat high   (rare words weighted up)
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print('Setup OK')


Setup OK


## 7. Bag of Words

CountVectorizer builds a vocabulary and a sparse document-term matrix.


In [2]:
docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are friends",
]

bow = CountVectorizer()
X_bow = bow.fit_transform(docs)

print("Vocabulary:", bow.get_feature_names_out())
print("Matrix shape:", X_bow.shape)
print("Dense:")
print(X_bow.toarray())
print("\nRow 0 ('the cat sat on the mat') counts:", X_bow.toarray()[0])


Vocabulary: ['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'friends' 'log' 'mat' 'on' 'sat'
 'the']
Matrix shape: (3, 12)
Dense:
[[0 0 1 0 0 0 0 0 1 1 1 2]
 [0 0 0 0 1 0 0 1 0 1 1 2]
 [1 1 0 1 0 1 1 0 0 0 0 0]]

Row 0 ('the cat sat on the mat') counts: [0 0 1 0 0 0 0 0 1 1 1 2]


## 8. TF-IDF by Hand

TF(t,d) = count of t in d. IDF(t) = log(N / df(t)). Score = TF x IDF (normalized). We verify against sklearn.


In [3]:
import math

def idf_by_hand(docs, term):
    N = len(docs)
    df = sum(1 for d in docs if term in d.split())
    return math.log((1 + N) / (1 + df)) + 1  # smooth, matches sklearn default

for term in ['the', 'cat', 'log', 'friends']:
    print(f"IDF('{term}') = {idf_by_hand(docs, term):.4f}")

tfidf = TfidfVectorizer(smooth_idf=True)
X = tfidf.fit_transform(docs)
print("\nsklearn IDF vector:")
for term in ['the', 'cat', 'log', 'friends']:
    if term in tfidf.vocabulary_:
        idx = tfidf.vocabulary_[term]
        print(f"  IDF('{term}') = {tfidf.idf_[idx]:.4f}")

# 'the' appears in every doc -> lowest IDF; rare terms (log, friends) -> high IDF
print("\nRare words get high IDF ('cat' appears once per doc but 'log'/'friends' rarely).")


IDF('the') = 1.2877
IDF('cat') = 1.6931
IDF('log') = 1.6931
IDF('friends') = 1.6931



sklearn IDF vector:
  IDF('the') = 1.2877
  IDF('cat') = 1.6931
  IDF('log') = 1.6931
  IDF('friends') = 1.6931

Rare words get high IDF ('cat' appears once per doc but 'log'/'friends' rarely).


## 9. Compare BoW vs TF-IDF Scores

See how common words drop and rare words rise.


In [4]:
common = 'the'
rare = 'friends'

for term in [common, rare]:
    b_idx = list(bow.vocabulary_).index(term) if term in bow.vocabulary_ else None
    t_idx = tfidf.vocabulary_.get(term)
    print(f"'{term}':")
    if b_idx is not None:
        print(f"  BoW count row2 = {X_bow.toarray()[2, b_idx]}")
    if t_idx is not None:
        print(f"  TF-IDF row2    = {X.toarray()[2, t_idx]:.4f}")

print("\n'friends' is rare -> TF-IDF boosts it; 'the' is everywhere -> suppressed.")


'the':
  BoW count row2 = 1
  TF-IDF row2    = 0.0000
'friends':
  BoW count row2 = 0
  TF-IDF row2    = 0.4472

'friends' is rare -> TF-IDF boosts it; 'the' is everywhere -> suppressed.


## 10. Data Leakage: Fit on Train, Transform on Test

Never fit the vectorizer on all data before splitting. Fit on train only, then `transform` test.


In [5]:
from sklearn.model_selection import train_test_split

corpus = ["I love python", "python is great for science",
          "science uses python a lot", "hate this boring text",
          "this course is amazing and fun", "worst tutorial i ever watched"]
y = [1, 1, 1, 0, 1, 0]
X_tr_text, X_te_text, y_tr, y_te = train_test_split(corpus, y, test_size=0.33,
                                                    random_state=0, stratify=y)

vec = TfidfVectorizer()
X_tr = vec.fit_transform(X_tr_text)   # fit ONLY on train
X_te = vec.transform(X_te_text)       # transform test with same vocab

print("Train shape:", X_tr.shape)
print("Test shape :", X_te.shape)
print("Both share the same vocabulary.")
print("\nIf we instead fit_transform on test, shapes misalign -> that is data leakage.")


Train shape: (4, 12)
Test shape : (2, 12)
Both share the same vocabulary.

If we instead fit_transform on test, shapes misalign -> that is data leakage.


## 11. Tuning Vocabulary Size

`max_features`, `min_df`, `max_df` control vocabulary size and memory use.


In [6]:
bigger = [
    "the quick brown fox jumps over the lazy dog",
    "a quick red squirrel climbs the tall tree",
    "the lazy dog sleeps under the brown tree",
    "two quick foxes race across the field quickly",
]

for mf in [3, 5, 10]:
    v = TfidfVectorizer(max_features=mf)
    M = v.fit_transform(bigger)
    print(f"max_features={mf}: vocab size={len(v.vocabulary_)}, shape={M.shape}")

v_min = TfidfVectorizer(min_df=2)  # keep words in >=2 docs
print("\nmin_df=2 keeps:", v_min.fit(bigger).get_feature_names_out())


max_features=3: vocab size=3, shape=(4, 3)
max_features=5: vocab size=5, shape=(4, 5)
max_features=10: vocab size=10, shape=(4, 10)

min_df=2 keeps: ['brown' 'dog' 'lazy' 'quick' 'the' 'tree']


## 12. Failure Case: Shape Mismatch From Data Leakage

Fitting a fresh vectorizer on test data creates a different vocabulary -> classification fails.


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# CORRECT: pipeline fit on train, predict on test
pipe = Pipeline([('tfidf', TfidfVectorizer()),
                 ('clf', LogisticRegression(max_iter=1000))])
pipe.fit(X_tr_text, y_tr)
print("Correct pipeline predict:", pipe.predict(X_te_text))

# WRONG: would fail if test had unseen words -> unknown vocab
try:
    bad = TfidfVectorizer().fit_transform(["totally brand new unseen words here"])
    print("\nSingle-doc vocab:", len(bad.toarray()[0]))
except Exception as e:
    print("\nError if vectorizer mismatched:", e)
print("\nLesson: always reuse the fitted vectorizer on test time.")


Correct pipeline predict: [1 1]

Single-doc vocab: 6

Lesson: always reuse the fitted vectorizer on test time.


## 13. Debugging: Common Errors

- **Memory error on large corpus** — vocabulary too big. Fix: `max_features`, `min_df`.
- **Test matrix different shape** — vocab mismatch. Fix: use `transform`, not `fit_transform`.
- **Features all zeros for a document** — no vocab words matched. Fix: check preprocessing consistency.
- **All IDF values ~1.0** — every word in every doc. Fix: diversify corpus, tweak `min_df`.

## 14. Real-World Considerations

- BoW/TF-IDF lose all word order (no sequence info) — a big limitation.
- Tune `max_features` (5k-50k), `min_df` (2-5), `max_df` (0.8-0.95).
- Save the fitted vectorizer with your model for consistent inference.

## 15. Common Mistakes

- Fitting the vectorizer on the entire dataset (data leakage).
- Ignoring vocabulary size / memory.
- Using BoW when word order matters.

## 16. When NOT to Use

- When you need semantic similarity or word meaning (use embeddings).
- When word order is critical (use sequence models / n-grams).

## 17. Challenge

Use the `analyzer='char_wb'` option to build character-level TF-IDF and compare with word-level.


In [8]:
# Challenge: character-level TF-IDF
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), max_features=15)
Xc = char_vec.fit_transform(bigger)
chars = char_vec.get_feature_names_out()
print("Char n-gram vocab (top 15):", chars[:15])
print("\nCharacter TF-IDF is robust to typos and splits words at boundaries ('_').")


Char n-gram vocab (top 15): [' qu' ' t' ' th' 'ck' 'e ' 'he' 'he ' 'ic' 'ick' 'qu' 'qui' 's ' 'th'
 'the' 'ui']

Character TF-IDF is robust to typos and splits words at boundaries ('_').


## 18. Closed-Book Recall

1. What does document-term matrix store?
2. Why does TF-IDF outperform raw counts for many tasks?
3. What is data leakage in vectorization and how to avoid it?
4. Which hyperparameters control vocabulary size?

## 19. Teach-Back Questions

- Explain TF-IDF to a colleague; why is 'the' low and a rare keyword high?
- Show the correct train/test vectorization flow.

## 20. Summary

You built BoW and TF-IDF matrices by hand and with sklearn, verified IDF math, tuned vocabulary, and avoided data leakage. These features feed classifiers in 07.4.

## 21. Further Experiment

- Compare word vs character TF-IDF on a typo-heavy classification task.
- Try HashingVectorizer for a streaming corpus.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
